# Calibrating a fly's lever arm

For most work you do not need this notebook — one command does the same job and
writes the result into the experiment YAML:

```bash
python -m kinefly2.apps.calibrate_fly <recording_dir> --write experiments/today.yaml
```

Use the notebook when you want to look at the fit rather than just accept it:
check the scatter, try different epochs, or compare flies.

## What is being fitted

The **front** camera measures head **roll** exactly (an in-plane rotation,
needing no rotation centre and no scale) and sees **pitch** and **yaw** only
end-on, as a shift of the head in pixels. One constant — the depth of the
landmark centroid from the neck along the optical axis — converts both. It
changes with head size, mounting and landmark placement, so it is **per fly**.

The **bottom** camera measures **yaw** exactly, and that is the reference. It
reports no displacement channels of its own: its lever arm (dorsoventral depth)
is too short, and those channels measured noise.

No stimulus is needed — spontaneous head movement is usually plenty.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from kinefly2.analysis import calibrate

LOG = 'logs/260819/260819_XHCS_017_YAMV/260819_XHCS_017_YAMV_flystate.h5'   # <- this fly
log = calibrate.load_log(LOG)
print(f"{len(log['merged/seq'])} rows")

for cam in ('bottom', 'front'):
    name, v = calibrate.exact_channel(log, cam)
    if name:
        print(f'{cam:7s} measures {name.upper():5s} exactly: std {np.degrees(np.nanstd(v)):.2f} deg')

## 1. Fit, and check whether the fit may be trusted

`check_fit` refuses recordings that cannot support a constant. A lever arm
fitted from a fly that barely moved is not a measurement but noise with units,
and it would silently rescale every angle derived from it.

In [ ]:
shift, reference, ref_name = calibrate.lever_inputs(log, 'front', reference='bottom')
# Optional: restrict to stimulus epochs once the arena is running.
# mask = calibrate.epochs_from_events(log, 'stim_on')
fit = calibrate.fit_lever(shift, reference)
ok, why = calibrate.check_fit(fit)
print(f'reference: bottom camera\'s exact {ref_name}\n')
print(calibrate.report(fit, ok, why))

In [ ]:
m = np.isfinite(shift) & np.isfinite(reference)
ref = reference[m] - np.median(reference[m])
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(np.degrees(ref), shift[m], '.', ms=1, alpha=.25)
if np.isfinite(fit['lever_px']):
    g = np.linspace(np.degrees(ref).min(), np.degrees(ref).max(), 100)
    plt.plot(g, fit['lever_px'] * np.sin(np.radians(g)) + fit['offset_px'], 'r-', lw=2)
plt.xlabel(f'bottom camera {ref_name} (deg)'); plt.ylabel('front head_d_lateral (px)')
plt.title(f"lever = {fit['lever_px']:.1f} px/rad, r2 = {fit['r2']:.3f}")

plt.subplot(1, 2, 2)
t = log['merged/t_host'][m]; t = t - t[0]
plt.plot(t, np.degrees(ref), lw=.7, label=f'bottom {ref_name} (deg)')
plt.plot(t, shift[m], lw=.7, label='front shift (px)')
plt.xlabel('time (s)'); plt.legend(); plt.title('do they move together?')
plt.tight_layout()

**If it was rejected**, the pixel channels are still valid — only the
conversion to angles is withheld. The usual cause is too little head movement:
record a fly that moves more, or drive it with an optomotor stimulus.

**Sanity check the number.** `lever_px` divided by your px/mm should land near a
few tenths of a millimetre — the neck-to-face depth. On the reference rig, 48.2
px at ~125 px/mm is 0.39 mm, which is right. Several millimetres would mean the
fit found something other than head rotation.

## 2. All three rotations

Yaw comes from the bottom camera, where it is exact. Roll comes from the front
camera, where it is exact. Pitch has no exact source and comes through the
lever arm — which is why the yaw comparison below matters: it is the same
constant, so if front yaw tracks bottom yaw, the pitch scale is trustworthy too.

In [ ]:
if ok:
    lever = fit['lever_px']
    yaw_bottom = np.degrees(log['bottom/head_yaw'])                       # exact
    roll_front = np.degrees(log['front/head_roll'])                       # exact
    yaw_front = np.degrees(calibrate.apply_lever(log['front/head_d_lateral'], lever))
    pitch = np.degrees(calibrate.apply_lever(log['front/head_d_axial'], lever))

    v = np.isfinite(yaw_front) & np.isfinite(yaw_bottom)
    r = np.corrcoef(yaw_front[v], yaw_bottom[v])[0, 1]
    slope = np.polyfit(yaw_front[v], yaw_bottom[v], 1)[0]
    print(f'yaw cross-check (front via lever vs bottom exact): r = {r:+.3f}, slope = {slope:.2f}')

    for name, x in (('yaw   (bottom, exact)', yaw_bottom),
                    ('roll  (front, exact)', roll_front),
                    ('pitch (front, lever)', pitch)):
        print(f'{name:24s} std {np.nanstd(x):5.2f} deg   '
              f'range {np.nanpercentile(x,1):+6.1f} .. {np.nanpercentile(x,99):+6.1f}')

    t = log['merged/t_host'] - log['merged/t_host'][0]
    fig, ax = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
    for a, (n, x) in zip(ax, (('yaw', yaw_bottom), ('roll', roll_front), ('pitch', pitch))):
        a.plot(t, x, lw=.6); a.set_ylabel(n + ' (deg)'); a.axhline(0, color='k', lw=.4)
    ax[-1].set_xlabel('time (s)'); plt.tight_layout()
else:
    print('Not calibrated - the pixel channels remain valid.')

## 3. Use it

Put the constant in **this fly's** experiment YAML — not the camera config,
where the next fly would inherit it:

```yaml
lever_px:
  front: 48.2
```

`calibrate_fly --write <experiment.yaml>` does exactly this, and additionally
checks that the landmarks have not moved since the recording and that
`axis_deg` points the right way. Prefer it unless you need the plots above.